Here is a complete **Google Colab-ready Lab Assignment** for Module 06.

This lab is designed to be **runnable on the free tier of Google Colab** (T4 GPU recommended, but many parts work on CPU). It covers the two critical pillars of Module 06: **Inference Control** (Decoding) and **Efficient Training** (PEFT/LoRA).

---

# 🧪 Lab 06: Taming the Model (Decoding & PEFT)

**Course:** Natural Language Processing (AI 2026)
**Module:** 06 - Advanced Text Generation & Summarization
**Estimated Time:** 60 Minutes
**Prerequisite:** A Google Account (to use Colab)

### 🎯 Learning Objectives
1.  **Master the "Chaos Knobs":** Understand how `Temperature`, `Top-P` (Nucleus), and `Beam Search` dramatically change AI output.
2.  **Summarization:** Use an Encoder-Decoder model (T5) to summarize dialogue.
3.  **Efficiency (LoRA):** Implement **Low-Rank Adaptation (LoRA)** to prepare a model for fine-tuning using only <1% of trainable parameters.

---

### 📝 Instructions for Students

1.  Open **Google Colab** (colab.research.google.com).
2.  Click **New Notebook**.
3.  Copy/Paste the code blocks below into cells.
4.  **Runtime Setup:** Go to `Runtime` > `Change runtime type` > Select **T4 GPU** (Recommended for Part 2).

---


# Part 1: The Art of Decoding (Controlling Output)
In this section, we use a lightweight model (distilgpt2) to simulate a robot generating a mission log. We will tweak the decoding strategies to see how the "personality" of the robot changes.

In [1]:
# INSTALL DEPENDENCIES
!pip install transformers torch peft datasets bitsandbytes -q
print("Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 20.5 MB/s eta 0:00:00
Dependencies installed!


In [2]:
# LOAD MODEL FOR GENERATION
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# We use DistilGPT2 for speed, but this works with Llama-3 or GPT-2-XL too
model_name = "distilgpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# The Scenario: A Robot in a dangerous situation
input_text = "The autonomous rover entered the dark cave and sensors detected"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

print(f"\nCONTEXT: '{input_text}...'")

Loading distilgpt2 on cuda...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


CONTEXT: 'The autonomous rover entered the dark cave and sensors detected...'


In [3]:
# EXPERIMENT 1: GREEDY SEARCH vs BEAM SEARCH
print("--- 1. GREEDY SEARCH (The Boring Robot) ---")
# Greedy picks the highest probability word every time. It is repetitive.
output_greedy = model.generate(input_ids, max_new_tokens=50, do_sample=False)
print("GREEDY:", tokenizer.decode(output_greedy[0], skip_special_tokens=True))

print("\n--- 2. BEAM SEARCH (The Careful Robot) ---")
# Beam search explores multiple paths (num_beams=5) to find the most 'likely' total sentence.
# Great for translation or summarization where accuracy matters more than creativity.
output_beam = model.generate(input_ids, max_new_tokens=50, num_beams=5, early_stopping=True)
print("BEAM:  ", tokenizer.decode(output_beam[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- 1. GREEDY SEARCH (The Boring Robot) ---


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


GREEDY: The autonomous rover entered the dark cave and sensors detected the presence of a large, dark-red object.








































--- 2. BEAM SEARCH (The Careful Robot) ---
BEAM:   The autonomous rover entered the dark cave and sensors detected it.















In [4]:
# EXPERIMENT 2: SAMPLING (Temperature & Top-P)
print("--- 3. HIGH TEMPERATURE (The Hallucinating Robot) ---")
# Temperature > 1.0 flattens the probability curve, making rare words more likely.
output_temp = model.generate(
    input_ids,
    max_new_tokens=50,
    do_sample=True,
    temperature=2.0,
    top_k=0
)
print("TEMP 2.0:", tokenizer.decode(output_temp[0], skip_special_tokens=True))

print("\n--- 4. NUCLEUS SAMPLING (The Modern Standard) ---")
# Top-P (Nucleus) samples from the dynamic set of tokens that make up the top 90% (0.9) probability.
# This balances creativity and coherence.
output_p = model.generate(
    input_ids,
    max_new_tokens=50,
    do_sample=True,
    top_p=0.90,
    temperature=0.7
)
print("NUCLEUS:", tokenizer.decode(output_p[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


--- 3. HIGH TEMPERATURE (The Hallucinating Robot) ---


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


TEMP 2.0: The autonomous rover entered the dark cave and sensors detected grounded band Canaveral Highway 41 beer king Ariel red contortion rings. SutTwo moves topple my barley margins tape gained snowy hairettair scores relying notcentral waken EkOTAppresses elimô throws Estonnik Archive shovecok Moinar Oswald reveals ever

--- 4. NUCLEUS SAMPLING (The Modern Standard) ---
NUCLEUS: The autonomous rover entered the dark cave and sensors detected the presence of a large boulder, which is larger than the Earth's surface.




































# Part 2: Abstractive Summarization with T5
Now we switch tasks. Instead of generating text from scratch, we will summarize a conversation. We will use FLAN-T5, an Encoder-Decoder model.

In [5]:
# LOAD T5 FOR SUMMARIZATION
from transformers import AutoModelForSeq2SeqLM

# Google's FLAN-T5 is excellent for instruction following and summarization
model_id = "google/flan-t5-base"

print(f"Loading {model_id}...")
tokenizer_t5 = AutoTokenizer.from_pretrained(model_id)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

# Dataset: A dialogue between a customer and support
dialogue = """
Customer: Hi, I bought a robot vacuum from your store yesterday, model XJ-9.
Support: Hello! Thank you for calling Robotics Inc. How can I help you with the XJ-9?
Customer: It keeps spinning in circles and shouting "Danger". I've tried resetting it.
Support: That sounds like a gyroscope calibration error.
Customer: How do I fix that?
Support: You need to hold the 'Home' button for 10 seconds until it beeps twice.
Customer: Okay, I will try that. Thanks.
"""

# Prompt Engineering for T5
prompt = f"Summarize the following conversation:\n{dialogue}\nSummary:"

inputs = tokenizer_t5(prompt, return_tensors="pt").to(device)

print("--- BASELINE SUMMARY (Zero-Shot) ---")
outputs = model_t5.generate(inputs["input_ids"], max_new_tokens=50)
print(tokenizer_t5.decode(outputs[0], skip_special_tokens=True))

Loading google/flan-t5-base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- BASELINE SUMMARY (Zero-Shot) ---
Customer: Hi, I'm looking for a repair for the XJ-9.


# Part 3: Parameter-Efficient Fine-Tuning (LoRA)
This is the "Engineering" part. Fine-tuning the full T5-Base model requires updating 250 Million parameters. This is slow and heavy.

We will use PEFT (Parameter-Efficient Fine-Tuning) to inject LoRA (Low-Rank Adaptation) adapters. This allows us to train the model by updating only 0.3% to 1% of the weights.

In [6]:
# SETUP LORA CONFIG
from peft import LoraConfig, get_peft_model, TaskType

# 1. Define the LoRA Configuration
lora_config = LoraConfig(
    r=8,                     # Rank: The size of the "low rank" matrix. Lower = fewer params.
    lora_alpha=32,           # Scaling factor.
    target_modules=["q", "v"], # Apply LoRA to Query and Value attention layers (Standard for Transformers)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # Tell PEFT this is for T5 (Seq2Seq), not GPT (Causal)
)

# 2. Inject LoRA into the Base Model
# This wraps the original model layers with the Adapter layers
peft_model = get_peft_model(model_t5, lora_config)

# 3. The "Aha!" Moment: Verify Parameter Reduction
print("\n--- PARAMETER COUNT COMPARISON ---")
peft_model.print_trainable_parameters()

# Expected Output:
# "trainable params: ~884,736 || all params: ~248,000,000 || trainable%: 0.35%"
# You are now ready to train this model on a single small GPU!


--- PARAMETER COUNT COMPARISON ---
trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## Part 4: Training the PEFT Model with LoRA

Now that we have injected LoRA adapters into our T5 model, we can proceed to train it. For this demonstration, we'll create a small dummy dataset. In a real-world scenario, you would load a larger summarization dataset (e.g., CNN/DailyMail, XSum).

In [7]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer

# 1. Prepare a dummy dataset for training
# In a real scenario, you'd load a dataset like 'samsum'

dummy_dialogues = [
    "Customer: My internet is not working. Support: Have you tried restarting your router? Customer: Yes, it didn't help. Support: Please describe the issue in more detail.",
    "User: I want to book a flight to London. Agent: When would you like to travel? User: Next month, any date. Agent: Let me check availability.",
    "Alice: I need to buy groceries. Bob: What do you need? Alice: Milk, eggs, and bread. Bob: I'll come with you."
]

dummy_summaries = [
    "Customer's internet is not working, and support suggests restarting the router and then asks for more details.",
    "User wants to book a flight to London next month, and the agent is checking availability.",
    "Alice needs to buy groceries (milk, eggs, bread), and Bob offers to join her."
]

# Combine into a dictionary to create a dataset
dummy_data = {'dialogue': dummy_dialogues, 'summary': dummy_summaries}
dataset = Dataset.from_dict(dummy_data)

def preprocess_function(examples):
    # This is similar to the prompt engineering we did earlier
    inputs = [f"summarize the following conversation:\n{d}\nSummary:" for d in examples["dialogue"]]
    model_inputs = tokenizer_t5(inputs, max_length=512, truncation=True, padding="max_length")

    # Setup the tokenizer for targets (summaries)
    labels = tokenizer_t5(text_target=examples["summary"], max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

print("Tokenized Dataset:", tokenized_dataset)

# Data Collator for Seq2Seq models
data_collator = DataCollatorForSeq2Seq(tokenizer_t5, model=peft_model)


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenized Dataset: Dataset({
    features: ['dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


### Define Training Arguments and Trainer

We'll now set up the `TrainingArguments` which specify our training parameters (e.g., number of epochs, learning rate, output directory). Then, we'll create a `Trainer` instance with our PEFT model, tokenized dataset, and data collator.

In [8]:
# 2. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./lora_t5_results",
    per_device_train_batch_size=1, # Small batch size for demonstration
    num_train_epochs=3, # Train for a few epochs
    learning_rate=2e-4,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none", # Avoids issues if wandb isn't set up
    use_cpu=device == "cpu" # Use CPU if CUDA is not available
)

# 3. Create Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer_t5,
    data_collator=data_collator,
)

print("Trainer created. Ready to train!")

Trainer created. Ready to train!


/tmp/ipython-input-1717534859.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start Training!

Now, let's kick off the training process. Since our dummy dataset is very small, this should complete quickly.

In [9]:
# 4. Train the model
trainer.train()

print("Training complete!")

# You can save the trained adapter weights
peft_model.save_pretrained("lora_t5_adapter")
print("LoRA adapter weights saved to 'lora_t5_adapter'/")

Step,Training Loss


Training complete!
LoRA adapter weights saved to 'lora_t5_adapter'/


### Test the Fine-Tuned Model

Let's see if our fine-tuned LoRA model produces a better summary for one of the training examples.

In [12]:
print("--- FINE-TUNED SUMMARY (LoRA) ---")

# Use the first dialogue from our dummy dataset
input_dialogue_ft = dummy_dialogues[0]
input_prompt_ft = f"summarize the following conversation:\n{input_dialogue_ft}\nSummary:"

inputs_ft = tokenizer_t5(input_prompt_ft, return_tensors="pt").to(device)

# Fix: Explicitly pass input_ids as a keyword argument
outputs_ft = peft_model.generate(input_ids=inputs_ft["input_ids"], max_new_tokens=50)
print(tokenizer_t5.decode(outputs_ft[0], skip_special_tokens=True))

print("\nOriginal Summary:", dummy_summaries[0])

--- FINE-TUNED SUMMARY (LoRA) ---
Customer: Your router is not working.

Original Summary: Customer's internet is not working, and support suggests restarting the router and then asks for more details.


# How to improve the LoRA model's summary quality

Improving the summary quality of a LoRA-tuned model involves several factors, ranging from data preparation to hyperparameter tuning. Here are the key ways to achieve better results:

More and Diverse Training Data: This is by far the most impactful factor. Our current example uses a tiny dummy dataset (3 examples). To get good summarization, you'd need hundreds to thousands of high-quality (dialogue, summary) pairs. The data should cover the specific domain or style of summarization you want the model to perform.

High-Quality Data: Ensure your (dialogue, summary) pairs are well-aligned. The summaries should be concise, accurate, and truly representative of the dialogue content. Avoid noisy or poorly written examples.

LoRA Hyperparameter Tuning:

r (Rank): We used r=8. Increasing r (e.g., to 16, 32, or 64) allows the LoRA adapters to learn more complex adaptations, potentially leading to better performance at the cost of slightly more trainable parameters. Experiment with different values.
lora_alpha: This is a scaling factor for the LoRA weights. A higher lora_alpha gives more weight to the LoRA updates. It often scales with r.
target_modules: We targeted q (query) and v (value) attention layers. You could also experiment with adding k (key) and o (output) layers, or even feed-forward network layers if the base model architecture supports it. However, q and v are usually a good starting point for T5-like models.
Training Arguments Tuning:

learning_rate: Experiment with different learning rates (e.g., 5e-5, 1e-4, 5e-4). LoRA often benefits from slightly higher learning rates than full fine-tuning.
num_train_epochs: We used 3 epochs. For a larger dataset, you'd typically need more epochs, but be careful not to overfit.
per_device_train_batch_size: Increase this if your GPU memory allows, as larger batch sizes can sometimes lead to more stable training.
Base Model Selection: While FLAN-T5-Base is good, starting with a larger or more specialized pre-trained model (e.g., FLAN-T5-Large, or a model specifically pre-trained for summarization) can provide a stronger foundation for LoRA to build upon.

Prompt Engineering: The way you phrase the prompt to the model (e.g., "summarize the following conversation:\n{dialogue}\nSummary:") can significantly influence the output. Experiment with different instructions and delimiters.

Decoding Strategies: After fine-tuning, the way you generate text (temperature, top_p, num_beams) still matters for the final output quality. For summarization, a balance between creativity and faithfulness is often desired. Beam search with a suitable num_beams can help generate more coherent and relevant summaries, especially in formal contexts.

By systematically experimenting with these aspects, you can significantly improve the performance of your LoRA-tuned summarization model.


## **Part 5: Conclusion & Lab Challenge**

**(Self-Guided for Students)**

1.  **Decoding Challenge:** Go back to Part 1. Change the `input_text` to: `"The chef opened the mystery box and found"`. Adjust the `temperature` to **2.5**. What happens to the language? Does it start inventing words?
2.  **LoRA Challenge:** In Part 3, change the `r` (Rank) from **8** to **32**. Re-run the cell. How does the number of trainable parameters change? (A higher rank means the adapter is "smarter" but larger).

### **Submission**
Export this notebook as `.ipynb` and submit it. Ensure the output of **Cell 6** (Parameter Count) is visible.